# **Define LLM from openrouter**

---



In [ ]:
!pip install -qU langchain "langchain[openai]"
!pip install langchain-classic
!pip install -U langchain-community
!pip install python-dotenv

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0.8
)

# **Data Loading (CV, TXT, LInkedIn)**

Note: LinkedIn blocks scraping so we will download it manully and parse it.

In [ ]:
!pip install pypdf

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

uploaded = files.upload()

for filename in uploaded.keys():

    with open(f"data/{filename}", "wb") as f:
        f.write(uploaded[filename])

Saving life_story.txt to life_story (1).txt
Saving Mariam-Ahmed-NLP-Engineer (1).pdf to Mariam-Ahmed-NLP-Engineer (1) (1).pdf
Saving Profile.pdf to Profile (1).pdf


In [ ]:
docs = load_documents("data")

In [ ]:
docs[0]

Document(metadata={'source': 'data/life_story.txt'}, page_content='my name is mariam , I live in Alexandria ')

In [ ]:
import re

def clean_text(text):

    text = re.sub(r'\s+', ' ', text)   # remove extra spaces
    text = re.sub(r'\n+', '\n', text)  # normalize new lines

    return text.strip()

In [ ]:
def linkedin_structure(text):

    sections = {
        "contact": "",
        "skills": "",
        "experience": "",
        "education": ""
    }

    current = "experience"

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if "Contact" in line:
            current = "contact"

        elif "Top Skills" in line:
            current = "skills"

        elif "Experience" in line:
            current = "experience"

        elif "Education" in line:
            current = "education"

        sections[current] += line + " "

    return sections

In [ ]:
from langchain_core.documents import Document

def to_documents(sections):

    docs = []

    for section, value in sections.items():

        if value.strip():

            docs.append(
                Document(
                    page_content=value,
                    metadata={"source": "linkedin", "section": section}
                )
            )

    return docs

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
import os


def load_all_data(data_folder="data"):

    all_docs = []

    for file in os.listdir("data"):

        path = os.path.join("data", file)

        # TXT FILES (CV / LIFE STORY)
        if file.endswith(".txt"):

            loader = TextLoader(path, encoding="utf-8")
            all_docs.extend(loader.load())

        # PDF FILES (LinkedIn)
        elif file.endswith(".pdf"):

            loader = PyPDFLoader(path)
            pdf_docs = loader.load()

            full_text = " ".join([d.page_content for d in pdf_docs])

            cleaned = clean_text(full_text)

            structured = linkedin_structure(cleaned)

            linkedin_docs = to_documents(structured)

            all_docs.extend(linkedin_docs)

    return all_docs

In [ ]:
docs = load_all_data()

print(len(docs))
print(docs[2].page_content)

6
Mariam Ahmed AbdelHaleem Mohammed Address: 5 57 St. Miami, Alexandria – Egypt Phone: +20-1062-57-3422 E-mail: mariam2500ahmed@gmail.com GitHub: https://github.com/MariamAhmedAbdElhaleem Career Objective Pursuing a rewarding career in the field of Natural Language Processing (NLP), where my technical skills, analysis skills and teamwork abilities can be used and developed. Education 2017-Present: M.S Candidate at institute of graduate studies and research, informa tio n technology department, Alexandria University. Thesis: “Applying Deep Learning Techniques to Build an Arabic Chatbot” 2015-2016: Information Technology Institute (ITI) 9 months Diploma Scholarship, Open Source Track, Alexandria Branch. 2011-2015: Bachelor of Arts, Alexandria University, Phonetics & Linguistics Department. Grade: Very good with the honors (B+), (Top 3 in Class). Graduation Project: Aggregation Sentences in Arabic using python script (Excellent grade) Other Projects: Arabic Morphological Analyzer using py

# **Chunking**

In [ ]:
# Chunking
from langchain_classic.text_splitter import TokenTextSplitter

splitter = TokenTextSplitter(
        chunk_size = 100,
        chunk_overlap = 10
)

chunks = splitter.split_documents(docs)

In [ ]:
chunks

[Document(metadata={'source': 'data/life_story.txt'}, page_content='my name is mariam , I live in Alexandria '),
 Document(metadata={'source': 'linkedin', 'section': 'contact'}, page_content="Contact mariam2500ahmed@gmail.com www.linkedin.com/in/mariam- shaheen-nlp (LinkedIn) Top Skills Language Modeling Natural Language Processing (NLP) Software Development Mariam Shaheen Natural Language Processing Specialist, Master's Student in NLP Egypt Experience Bibliotheca Alexandrina Natural Language Processing Tools Developer November 2017 - Present (8 years 8 months) Alexandria Governorate, Egypt • Developing a word sense disambiguation"),
 Document(metadata={'source': 'linkedin', 'section': 'contact'}, page_content=' • Developing a word sense disambiguation tool that automatically predict the best lexical semantic in Arabic language. annotation for ICA (A project that contains 100 million Arabic words that are planned to be analyzed morphologically, syntactically and semantically). • Revamp

# **Embeding**

In [ ]:
# Embedding
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs = {"device":"cpu"}
)

vectors = embeddings.embed_documents([i.page_content for i in chunks])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# FAISS vector store

In [ ]:
!pip install faiss-cpu

In [ ]:
# vector store
from langchain_classic.vectorstores import FAISS

vectorDB = FAISS.from_documents(chunks,embeddings)

In [ ]:
def retrieve_context(vectorDB, query):

    docs = vectorDB.similarity_search(query, k=4)

    # Combine retrieved text
    context = "\n".join([d.page_content for d in docs])

    return context

In [ ]:
query = "Who is this person?"
context = retrieve_context(vectorDB, query)

In [ ]:
context

'Mariam Ahmed AbdelHaleem Mohammed Address: 5 57 St. Miami, Alexandria – Egypt Phone: +20-1062-57-3422 E-mail: mariam2500ahmed@gmail.com GitHub: https://github.com/MariamAhmedAbdElhaleem Career Objective Pursuing a rewarding career in the field of Natural Language Processing (NLP), where my technical skills, analysis skills and teamwork abilities can be used and developed. Education 2017-Present\nMariam Ahmed AbdelHaleem Mohammed Address: 5 57 St. Miami, Alexandria – Egypt Phone: +20-1062-57-3422 E-mail: mariam2500ahmed@gmail.com GitHub: https://github.com/MariamAhmedAbdElhaleem Career Objective Pursuing a rewarding career in the field of Natural Language Processing (NLP), where my technical skills, analysis skills and teamwork abilities can be used and developed. Education 2017-Present\n memory in “Omdena” (2022: present) \uf0d8 Participant at International Corpus of Arabic (text classification and morphological analysis). \uf0d8 Member of students union at the Alexandria University, 

# **Memory**

In [ ]:
# define converstion
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

chat_memory = ConversationChain(
    llm = llm,
    memory = ConversationBufferMemory(
        memory_key="history", #Defines the dictionary key used to pass the saved dialogue
        return_messages=True #Extracts past conversation turns as distinct objects (like HumanMessage and AIMessage) rather than flattening them into a raw text string.
    )
)

response = chat_memory.predict(input="who Iam ?")

In [ ]:
def get_history():

    return chat_memory.memory.load_memory_variables({})["history"]

In [ ]:
response

'Hello! I’m an AI language model, so I don’t have direct access to personal data about you unless you share it with me in this conversation. Since you haven’t told me anything about yourself yet, I don’t actually know who you are. \n\nIf you’d like, feel free to tell me a bit about yourself—your name, interests, what you’re working on, or anything else you’d like to share—and I’ll gladly keep the conversation going! 🌟'

# **prompt Engineering**

In [ ]:
from langchain_core.prompts import PromptTemplate

temp = """
you are assistant, Answer the next Question using provided context,
If you don't know the answer, just say you don't know.
answer should be within 30 words or lower only.
Answer in {language}.

## context :
{context}

## Chat History:
{history}

## Question :
{Question}

"""

temp = PromptTemplate.from_template(temp)


query1 = "Who is this person?"
query2= "What are his skills?"
query3= "What projects did he work on?"
query4= "What are his interests?"
query5= "Summarize his career."
history = get_history()
context = retrieve_context(vectorDB, query)



prompt = temp.format(
    context=context,
    Question=query1,
    history=history,
    language="arabic"
)
print(prompt)


you are assistant, Answer the next Question using provided context,
If you don't know the answer, just say you don't know.
answer should be within 30 words or lower only.
Answer in arabic.

## context :
Mariam Ahmed AbdelHaleem Mohammed Address: 5 57 St. Miami, Alexandria – Egypt Phone: +20-1062-57-3422 E-mail: mariam2500ahmed@gmail.com GitHub: https://github.com/MariamAhmedAbdElhaleem Career Objective Pursuing a rewarding career in the field of Natural Language Processing (NLP), where my technical skills, analysis skills and teamwork abilities can be used and developed. Education 2017-Present
Mariam Ahmed AbdelHaleem Mohammed Address: 5 57 St. Miami, Alexandria – Egypt Phone: +20-1062-57-3422 E-mail: mariam2500ahmed@gmail.com GitHub: https://github.com/MariamAhmedAbdElhaleem Career Objective Pursuing a rewarding career in the field of Natural Language Processing (NLP), where my technical skills, analysis skills and teamwork abilities can be used and developed. Education 2017-Present


In [ ]:
response = llm.invoke(prompt).content

In [ ]:
from IPython.display import display , Markdown

display(Markdown(response))

مريم أحمد عبدالحليم محمد، مصرية الجنسية، طالبة بآداب الإسكندرية، تعمل في معالجة اللغات الطبيعية وتصنيف النصوص العربية، ومتطوعة بالاتحاد العربي الطبي.